In [1]:
import os
import sys
import pickle as pkl
import pandas as pd
import numpy as np
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm

/home/cyrus/venvs/uts-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import Dataset
Dataset contains SBERT question, human response, agent response, and agent summary vectors. 

In [2]:
sys.path.append(str(Path.cwd().parent))

ROOT = Path(os.getcwd()).resolve().parents[0]
DATA = ROOT / "data"

with open(DATA / "concat_train_set.pkl", "rb") as file:
    train_set = pkl.load(file)

with open(DATA / "concat_test_set.pkl", "rb") as file:
    test_set = pkl.load(file)

train_df    = pd.DataFrame(train_set)
test_df     = pd.DataFrame(test_set)

print("Train set:", train_df.shape)
print("Test set:", train_df.shape)

Train set: (1708, 12)
Test set: (1708, 12)


## Load BERT Models

In [3]:
import torch
from transformers import (
    BertForQuestionAnswering, 
    AutoTokenizer,
)
from sentence_transformers import SentenceTransformer

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [5]:
bert_model = BertForQuestionAnswering.from_pretrained("deepset/bert-base-cased-squad2")
bert_model.to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("deepset/bert-base-cased-squad2")
sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2').to(device)

# Count BERT's total number of trainable parameters by summing the size of every weight tensor.
bert_num_params = sum(p.numel() for p in bert_model.parameters())
# Print the count, so "small model" isn't just a label in a markdown cell -- see the real number.
print(f"BERT ({bert_model.config.name_or_path}) parameters: {bert_num_params:,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9861.02it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9113.05it/s]


BERT (deepset/bert-base-cased-squad2) parameters: 107,721,218


## Training Configuration

In [6]:
from dataclasses import dataclass

@dataclass
class Config:
    # Training config
    epochs: int         = 50

    # Data shape
    question_dim: int   = sentence_model.get_sentence_embedding_dimension()
    bert_dim: int       = sentence_model.get_sentence_embedding_dimension()

    # Latent sizes
    style_dim: int      = sentence_model.get_sentence_embedding_dimension()
    content_dim: int    = sentence_model.get_sentence_embedding_dimension()
    context_dim: int    = sentence_model.get_sentence_embedding_dimension()

    disc_dim: int       = sentence_model.get_sentence_embedding_dimension()
    gene_dim: int       = sentence_model.get_sentence_embedding_dimension()

    # Optimisation
    lr_generator: float     = 1e-4
    lr_discriminator: float = 1e-4
    batch_size: int         = 32

cfg = Config(epochs=50)

/tmp/ipykernel_108929/341774889.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  question_dim: int   = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_108929/341774889.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  bert_dim: int       = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_108929/341774889.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  style_dim: int      = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_108929/341774889.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  content_dim: int    = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_108929/341774889.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed 

In [7]:
from train.pairs import *

train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)
train_human     = torch.tensor(train_df["hr_bert"].to_list(), dtype=torch.float32)
train_agent     = torch.tensor(train_df["as_bert"].to_list(), dtype=torch.float32)

test_question   = torch.tensor(test_df["question_bert"].to_list(), dtype=torch.float32)
test_human      = torch.tensor(test_df["hr_bert"].to_list(), dtype=torch.float32)
test_agent      = torch.tensor(test_df["as_bert"].to_list(), dtype=torch.float32)

train_dataset = SummaryPairDataset(
    train_question,
    train_human,
    train_agent
)
test_dataset = SummaryPairDataset(
    test_question,
    test_human,
    test_agent
)

train_loader    = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
test_loader     = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=True) 

/tmp/ipykernel_108929/3805931428.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)


In [8]:
train_question.shape

torch.Size([1708, 384])

## Load Disentangler

In [9]:
from models.disentangler import SBERTDisentangler
from models.discriminator import SBERTDiscriminator
from models.generator import SBERTGenerator
from models.encoders import (
    StyleEncoder,
    ContentEncoder,
    ContextEncoder,
)
from train.step import *

In [10]:
def alter_train(model, optim_G, optim_D, loader, n_epochs):
    history = []

    state = AlternatingState()

    for epoch in range(n_epochs):
        model.train()

        epoch_logs = {}
        n_G, n_D = 0, 0

        for batch in tqdm(loader, desc=f"epoch {epoch + 1}/{n_epochs}", leave=False):
            batch = {key: value.to(device) for key, value in batch.items()}

            logs = train_iteration(
                model=model,
                batch=batch,
                optim_G=optim_G,
                optim_D=optim_D,
                state=state,
                lambda_cross=1.0,
                lambda_adv=1.0,
            )

            if logs["updated"] == "G":
                n_G += 1
            else:
                n_D += 1

            # Accumulate only values that exist for this update.
            for key, value in logs.items():
                if (isinstance(value, (int, float)) and np.isfinite(value)):
                    epoch_logs[key] = (epoch_logs.get(key, 0.0) + value)

        # Average G and D losses separately.
        mean_logs = {}

        for key, value in epoch_logs.items():
            if key.startswith("loss_D"):
                mean_logs[key] = value / max(n_D, 1)
            else:
                mean_logs[key] = value / max(n_G, 1)

        mean_logs["epoch"] = epoch + 1
        mean_logs["n_G"] = n_G
        mean_logs["n_D"] = n_D

        history.append(mean_logs)

        print(
            f"epoch {epoch + 1:3d} | "
            f"G updates {n_G:4d} | "
            f"D updates {n_D:4d} | "
            f"self {mean_logs.get('loss_self', float('nan')):.4f} | "
            f"cross {mean_logs.get('loss_cross', float('nan')):.4f} | "
            f"adv {mean_logs.get('loss_adv', float('nan')):.4f} | "
            f"D {mean_logs.get('loss_D', float('nan')):.4f}"
        )

    return model, history

In [11]:
def mixed_train(model, optim_G, optim_D, loader, n_epochs):
    history = []

    for epoch in range(n_epochs):
        epoch_losses = {}

        for batch in tqdm(train_loader, total=len(train_loader), desc="Training"):
            batch = {key: value.to(device) for key, value in batch.items()}
            losses = train_step(model, batch, optim_G, optim_D)

            for name, value in losses.items():
                epoch_losses[name] = (epoch_losses.get(name, 0.0) + value)

        for name in epoch_losses:
            epoch_losses[name] /= len(train_loader)
            
        history.append(epoch_losses)
        print(
            f"epoch {epoch + 1:3d} | "
            f"generator loss {epoch_losses['generator_loss']:.4f} | "
            f"self loss {epoch_losses['self_loss']:.4f} | "
            f"cross loss {epoch_losses['cross_loss']:.4f} | "
            f"adversarial loss {epoch_losses['adversarial_loss']:.4f} | "
            f"discriminator loss {epoch_losses['discriminator_loss']:.4f}"
        )

    return model, history

In [12]:
alter_model = SBERTDisentangler(cfg).to(device)
alter_optim_G = torch.optim.Adam(
    alter_model.generator_parameters(),
    lr=cfg.lr_generator
)
alter_optim_D = torch.optim.Adam(
    alter_model.discriminator_parameters(),
    lr=cfg.lr_discriminator
)

In [13]:
alter_model, alter_hist = alter_train(alter_model, alter_optim_G, alter_optim_D, train_loader, 100)

epoch   1 | G updates   27 | D updates   27 | self 7.0989 | cross 4.2478 | adv 0.6947 | D 0.6306


epoch   2 | G updates   15 | D updates   39 | self 6.7049 | cross 4.1567 | adv 0.6913 | D 0.4955


epoch   3 | G updates   15 | D updates   39 | self 6.6207 | cross 4.1274 | adv 0.8662 | D 0.5780


epoch   4 | G updates   18 | D updates   36 | self 6.6040 | cross 4.0794 | adv 1.0174 | D 0.6236


epoch   5 | G updates   18 | D updates   36 | self 6.7347 | cross 4.0884 | adv 0.7431 | D 0.5179


...

epoch  96 | G updates   21 | D updates   33 | self 6.1185 | cross 3.3017 | adv 0.9881 | D 0.6515


epoch  97 | G updates   18 | D updates   36 | self 6.1346 | cross 3.3025 | adv 1.0815 | D 0.6302


epoch  98 | G updates   21 | D updates   33 | self 6.0395 | cross 3.2472 | adv 0.9471 | D 0.6390


epoch  99 | G updates   18 | D updates   36 | self 6.1754 | cross 3.3234 | adv 1.0307 | D 0.6415


epoch 100 | G updates   22 | D updates   32 | self 6.1862 | cross 3.3101 | adv 1.1179 | D 0.6328


In [14]:
mixed_model = SBERTDisentangler(cfg).to(device)
mixed_optim_G = torch.optim.Adam(
    mixed_model.generator_parameters(),
    lr=cfg.lr_generator
)
mixed_optim_D = torch.optim.Adam(
    mixed_model.discriminator_parameters(),
    lr=cfg.lr_discriminator
)

In [15]:
mixed_model, mixed_hist = mixed_train(mixed_model, mixed_optim_G, mixed_optim_D, train_loader, cfg.epochs)

Training: 100%|██████████| 54/54 [00:00<00:00, 55.69it/s]


epoch   1 | generator loss 11.7085 | self loss 6.9306 | cross loss 4.2131 | adversarial loss 0.5647 | discriminator loss 0.6757


Training: 100%|██████████| 54/54 [00:01<00:00, 37.32it/s]


epoch   2 | generator loss 11.3761 | self loss 6.6135 | cross loss 4.0237 | adversarial loss 0.7389 | discriminator loss 0.7544


Training: 100%|██████████| 54/54 [00:01<00:00, 32.08it/s]


...

epoch  48 | generator loss 9.5037 | self loss 5.7924 | cross loss 2.9653 | adversarial loss 0.7460 | discriminator loss 0.6553


Training: 100%|██████████| 54/54 [00:01<00:00, 28.50it/s]


epoch  49 | generator loss 9.4816 | self loss 5.7925 | cross loss 2.9668 | adversarial loss 0.7223 | discriminator loss 0.6725


Training: 100%|██████████| 54/54 [00:01<00:00, 32.08it/s]

epoch  50 | generator loss 9.5139 | self loss 5.8100 | cross loss 2.9987 | adversarial loss 0.7053 | discriminator loss 0.6700


## Evaluation

In [16]:
@torch.no_grad()
def evaluate_encoders(model, val_loader, device):
    model.eval()

    all_questions = []
    all_human_style = []
    all_agent_style = []
    all_human_content = []
    all_agent_content = []
    all_human_context = []
    all_agent_context = []

    for batch in val_loader:
        question = batch["question"].to(device)
        human = batch["human"].to(device)
        agent = batch["agent"].to(device)

        # Encode the paired human and AI responses
        style_h, content_h, context_h = model.encode(human)
        style_a, content_a, context_a = model.encode(agent)

        all_questions.append(question.cpu())
        all_human_style.append(style_h.cpu())
        all_agent_style.append(style_a.cpu())
        all_human_content.append(content_h.cpu())
        all_agent_content.append(content_a.cpu())
        all_human_context.append(context_h.cpu())
        all_agent_context.append(context_a.cpu())

    # Combine batches so retrieval is evaluated across the full
    # validation set, rather than only within individual batches.
    questions = torch.cat(all_questions)
    style_h = torch.cat(all_human_style)
    style_a = torch.cat(all_agent_style)
    content_h = torch.cat(all_human_content)
    content_a = torch.cat(all_agent_content)
    context_h = torch.cat(all_human_context)
    context_a = torch.cat(all_agent_context)

    # ---------------------------------------------------------
    # 1. STYLE: human vs. AI classification accuracy
    # ---------------------------------------------------------
    style_embeddings = torch.cat([style_h, style_a], dim=0)

    style_labels = torch.cat([
        torch.zeros(len(style_h)),
        torch.ones(len(style_a)),
    ]).long()

    style_logits = model.style_classifier(
        style_embeddings.to(device)
    ).squeeze(-1).cpu()

    style_predictions = (style_logits >= 0).long()
    style_accuracy = (
        style_predictions == style_labels
    ).float().mean().item()

    # ---------------------------------------------------------
    # 2. CONTENT: paired-response retrieval accuracy
    # ---------------------------------------------------------
    # Each human response should match its paired AI response.
    # Normalize vectors so matrix multiplication gives cosine
    # similarity.
    content_h = F.normalize(content_h, p=2, dim=-1)
    content_a = F.normalize(content_a, p=2, dim=-1)

    content_similarity = content_h @ content_a.T
    targets = torch.arange(len(content_h))

    # Human -> AI and AI -> human retrieval
    h_to_a = (
        content_similarity.argmax(dim=1) == targets
    ).float().mean().item()

    a_to_h = (
        content_similarity.argmax(dim=0) == targets
    ).float().mean().item()

    content_accuracy = (h_to_a + a_to_h) / 2

    # Also report paired cosine similarity for interpretability.
    content_paired_cosine = (
        content_h * content_a
    ).sum(dim=-1).mean().item()

    # ---------------------------------------------------------
    # 3. CONTEXT: question embedding prediction
    # ---------------------------------------------------------
    # The context regressor predicts the original SBERT
    # question embedding from each context representation.
    predicted_context_h = model.context_regressor(
        context_h.to(device)
    ).cpu()

    predicted_context_a = model.context_regressor(
        context_a.to(device)
    ).cpu()

    predicted_context = torch.cat(
        [predicted_context_h, predicted_context_a], dim=0
    )

    # Each question is a candidate target for retrieval.
    # The question embedding is shared by its human and AI pair.
    question_candidates = F.normalize(questions, p=2, dim=-1)
    predicted_context = F.normalize(
        predicted_context, p=2, dim=-1
    )

    # Repeat question indices to align with human and AI contexts.
    context_targets = torch.cat([
        torch.arange(len(questions)),
        torch.arange(len(questions)),
    ])

    context_similarity = predicted_context @ question_candidates.T

    context_retrieval_accuracy = (
        context_similarity.argmax(dim=1) == context_targets
    ).float().mean().item()

    # Cosine similarity with the correct question for each response
    true_questions = questions.repeat(2, 1)
    context_cosine = F.cosine_similarity(
        predicted_context, true_questions, dim=-1
    ).mean().item()

    # ---------------------------------------------------------
    # Summary table
    # ---------------------------------------------------------
    results = pd.DataFrame([
        {
            "Encoder": "Style",
            "Metric": "Human/AI classification accuracy",
            "Score": style_accuracy,
        },
        {
            "Encoder": "Content",
            "Metric": "Paired retrieval accuracy (top-1)",
            "Score": content_accuracy,
        },
        {
            "Encoder": "Content",
            "Metric": "Paired cosine similarity",
            "Score": content_paired_cosine,
        },
        {
            "Encoder": "Context",
            "Metric": "Question retrieval accuracy (top-1)",
            "Score": context_retrieval_accuracy,
        },
        {
            "Encoder": "Context",
            "Metric": "Question cosine similarity",
            "Score": context_cosine,
        },
    ])

    results["Score"] = results["Score"].round(4)

    return results

In [17]:
device = next(alter_model.parameters()).device

results = evaluate_encoders(
    model=alter_model,
    val_loader=train_loader,
    device=device,
)

print("\nAlternating State Training Results")
print(results)

device = next(alter_model.parameters()).device

results = evaluate_encoders(
    model=alter_model,
    val_loader=test_loader,
    device=device,
)

print("\nAlternating State Testing Results")
print(results)


Alternating State Training Results
   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.8349
1  Content    Paired retrieval accuracy (top-1)  0.8996
2  Content             Paired cosine similarity  0.9220
3  Context  Question retrieval accuracy (top-1)  0.5828
4  Context           Question cosine similarity  0.7163

Alternating State Testing Results
   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.7737
1  Content    Paired retrieval accuracy (top-1)  0.9421
2  Content             Paired cosine similarity  0.9135
3  Context  Question retrieval accuracy (top-1)  0.7474
4  Context           Question cosine similarity  0.7128


In [18]:
device = next(mixed_model.parameters()).device

results = evaluate_encoders(
    model=mixed_model,
    val_loader=train_loader,
    device=device,
)

print("\nMixed State Training Results")
print(results)

device = next(mixed_model.parameters()).device

results = evaluate_encoders(
    model=mixed_model,
    val_loader=test_loader,
    device=device,
)

print("\nMixed State Testing Results")
print(results)


Mixed State Training Results
   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.8791
1  Content    Paired retrieval accuracy (top-1)  0.9561
2  Content             Paired cosine similarity  0.9231
3  Context  Question retrieval accuracy (top-1)  0.9177
4  Context           Question cosine similarity  0.8156

Mixed State Testing Results
   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.7816
1  Content    Paired retrieval accuracy (top-1)  0.9605
2  Content             Paired cosine similarity  0.9039
3  Context  Question retrieval accuracy (top-1)  0.9421
4  Context           Question cosine similarity  0.8016
